# 02 Pipeline

A compact, cloneable ETL: prep decides, physical IO executes, and checks plus metadata registration stay visible.

## Tested with FabricOps

The previous baseline was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesigned workflow has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Load the shared Fabric configuration, public APIs, and only the two registries needed across source blocks.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    profile_and_register_table,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_query,
    write_lakehouse_table,
    write_pipeline_prep,
    widget_select_data_contract,
    widget_view_catalogue,
)

SOURCE_PREPS = {}
SOURCE_DFS = {}
PIPELINE_SHOULD_RUN = True

## Notebook controls

Use one optional Data Contract selector and one Catalogue view. FabricOps uses the notebook name as the stable logical Lineage identity across environments; runtime notebook, workspace, and environment IDs remain audit context.

In [ ]:
# Step 2: leave False for the first run, before Lineage and frozen contracts exist.
# Step 4: set True to select one frozen contract for every lineage-linked table_id.
VALIDATE_DATA_CONTRACTS = False
CONTRACT_SELECTION = (
    widget_select_data_contract(spark_session=spark)
    if VALIDATE_DATA_CONTRACTS
    else None
)

catalogue_widget = widget_view_catalogue(mode="explore", spark_session=spark)

## How to read the blocks

Each block defines physical identity and strategy—not `table_id`. Prep deterministically resolves `table_id` and processing scope. The visible reader or writer then performs physical IO.

In [ ]:
# Orders uses this physical target only to resolve authoritative target-backed progress.
ORDERS_PROGRESS_TARGET = {
    "target_target": "unified",
    "target_schema": "demo",
    "target_table": "orders",
}

# E. Extract

Each source follows **define → prep → pre-read checks → physical read → data checks → profile/register → store → view**.

## SOURCE 1 — Orders

Incremental Lakehouse rows bounded by the successful target watermark.

In [ ]:
SOURCE = 1
SOURCE_NAME = "Orders"
SOURCE_TARGET = "source"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE = "orders"
SOURCE_READ_STRATEGY = "incremental_watermark"
SOURCE_WATERMARK_COLUMN = "modified_datetime"

In [ ]:
source_prep = read_pipeline_prep(
    source_target=SOURCE_TARGET,
    source_schema=SOURCE_SCHEMA,
    source_table=SOURCE_TABLE,
    source_read_strategy=SOURCE_READ_STRATEGY,
    source_watermark_column=SOURCE_WATERMARK_COLUMN,
    **ORDERS_PROGRESS_TARGET,
)
SOURCE_TABLE_ID = source_prep["table_id"]

# Pre-read checks: these use observation/metadata only and stay before physical IO.
pre_read_results = []
if VALIDATE_DATA_CONTRACTS and source_prep["observation"] is not None:
    pre_read_results.append(check_freshness(source_prep["observation"], table_id=SOURCE_TABLE_ID))
if source_prep["changes"] is not None:
    pre_read_results.append(source_prep["changes"])
if not all(result["can_continue"] for result in pre_read_results):
    raise RuntimeError(f"A SOURCE {SOURCE} pre-read Guardrail blocked this run.")

PIPELINE_SHOULD_RUN = source_prep["read_mode"] != "skip"
if PIPELINE_SHOULD_RUN:
    source_df = read_lakehouse_table(
        source_prep["source"]["table_name"],
        target=source_prep["source"]["target"],
        schema=source_prep["source"]["schema"],
        spark_session=spark,
        processing_scope=source_prep["scope"],
    )

    source_checks = (
        [
            check_schema(table_id=SOURCE_TABLE_ID, dataframe=source_df),
            check_dq(source_df, table_id=SOURCE_TABLE_ID),
        ]
        if VALIDATE_DATA_CONTRACTS
        else []
    )
    if source_checks:
        display(source_checks[-1]["summary"])
    if not all(result["can_continue"] for result in source_checks):
        raise RuntimeError(f"A SOURCE {SOURCE} data Guardrail blocked this run.")

    source_profile = profile_and_register_table(
        source_df,
        profile_role="source",
        table=source_prep["source"],
        processing_scope=source_prep["scope"],
    )
    display(source_profile)
    SOURCE_PREPS[SOURCE] = source_prep
    SOURCE_DFS[SOURCE] = source_df

    catalogue_widget["refresh"]()
    source_views = catalogue_widget["get_views"]()
    display(source_views["catalogue"].filter(F.col("table_id") == SOURCE_TABLE_ID))
    display(source_views["profile"].filter(F.col("table_id") == SOURCE_TABLE_ID))
    display(source_views["frequency"].filter(F.col("table_id") == SOURCE_TABLE_ID))
else:
    print("Orders is unchanged; physical reads, transforms, and writes are skipped.")

## SOURCE 2 — Products

A complete Lakehouse reference read with the same visible stages.

In [ ]:
SOURCE = 2
SOURCE_NAME = "Products"
SOURCE_TARGET = "source"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE = "products"
SOURCE_READ_STRATEGY = "full_dataset"

In [ ]:
if PIPELINE_SHOULD_RUN:
    source_prep = read_pipeline_prep(
        source_target=SOURCE_TARGET,
        source_schema=SOURCE_SCHEMA,
        source_table=SOURCE_TABLE,
        source_read_strategy=SOURCE_READ_STRATEGY,
    )
    SOURCE_TABLE_ID = source_prep["table_id"]

    # No row-free freshness/change check is configured for this full read.
    source_df = read_lakehouse_table(
        source_prep["source"]["table_name"],
        target=source_prep["source"]["target"],
        schema=source_prep["source"]["schema"],
        spark_session=spark,
        processing_scope=source_prep["scope"],
    )

    source_checks = (
        [
            check_schema(table_id=SOURCE_TABLE_ID, dataframe=source_df),
            check_dq(source_df, table_id=SOURCE_TABLE_ID),
        ]
        if VALIDATE_DATA_CONTRACTS
        else []
    )
    if source_checks:
        display(source_checks[-1]["summary"])
    if not all(result["can_continue"] for result in source_checks):
        raise RuntimeError(f"A SOURCE {SOURCE} data Guardrail blocked this run.")

    source_profile = profile_and_register_table(
        source_df,
        profile_role="source",
        table=source_prep["source"],
        processing_scope=source_prep["scope"],
    )
    display(source_profile)
    SOURCE_PREPS[SOURCE] = source_prep
    SOURCE_DFS[SOURCE] = source_df

    catalogue_widget["refresh"]()
    source_views = catalogue_widget["get_views"]()
    display(source_views["catalogue"].filter(F.col("table_id") == SOURCE_TABLE_ID))
    display(source_views["profile"].filter(F.col("table_id") == SOURCE_TABLE_ID))
    display(source_views["frequency"].filter(F.col("table_id") == SOURCE_TABLE_ID))

## SOURCE 3 — Order History

The SQL stays visible and executes in the Warehouse; its aggregate result receives a diagnostic profile.

In [ ]:
SOURCE = 3
SOURCE_NAME = "Order History"
SOURCE_TARGET = "product"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE = "order_history"
SOURCE_READ_STRATEGY = "full_dataset"

SOURCE_QUERY = """
SELECT
    customer_id,
    COUNT(*) AS historical_order_count,
    SUM(net_amount) AS historical_net_amount,
    MAX(order_datetime) AS latest_historical_order_datetime
FROM demo.order_history
GROUP BY customer_id
"""

In [ ]:
if PIPELINE_SHOULD_RUN:
    source_prep = read_pipeline_prep(
        source_target=SOURCE_TARGET,
        source_schema=SOURCE_SCHEMA,
        source_table=SOURCE_TABLE,
        source_read_strategy=SOURCE_READ_STRATEGY,
    )
    SOURCE_TABLE_ID = source_prep["table_id"]

    # No row-free freshness/change check is configured for this full query.
    source_df = read_warehouse_query(
        SOURCE_QUERY,
        target=source_prep["source"]["target"],
        spark_session=spark,
    )

    source_checks = (
        [
            check_schema(table_id=SOURCE_TABLE_ID, dataframe=source_df),
            check_dq(source_df, table_id=SOURCE_TABLE_ID),
        ]
        if VALIDATE_DATA_CONTRACTS
        else []
    )
    if source_checks:
        display(source_checks[-1]["summary"])
    if not all(result["can_continue"] for result in source_checks):
        raise RuntimeError(f"A SOURCE {SOURCE} data Guardrail blocked this run.")

    source_profile = profile_and_register_table(
        source_df,
        profile_role="source",
        table=source_prep["source"],
        processing_scope=source_prep["scope"],
        complete_table=False,
    )
    display(source_profile)
    SOURCE_PREPS[SOURCE] = source_prep
    SOURCE_DFS[SOURCE] = source_df

    catalogue_widget["refresh"]()
    source_views = catalogue_widget["get_views"]()
    display(source_views["catalogue"].filter(F.col("table_id") == SOURCE_TABLE_ID))
    display(source_views["profile"].filter(F.col("table_id") == SOURCE_TABLE_ID))
    display(source_views["frequency"].filter(F.col("table_id") == SOURCE_TABLE_ID))

### Profile semantics

Complete physical reads update canonical `METADATA_DATA_CATALOGUE`, `METADATA_DATA_PROFILED`, and eligible `METADATA_DATA_PROFILED_FREQUENCY` snapshots. Incremental subsets and custom query results return diagnostic profiles without replacing canonical full-table metadata. Source and target participation is linked in `METADATA_DATA_LINEAGE` when prep contexts meet at write preparation.

# T. Transform

**Business transformation is project-owned PySpark.** FabricOps does not hide these joins or calculations.

In [ ]:
if PIPELINE_SHOULD_RUN:
    orders_df = SOURCE_DFS[1].alias("orders")
    products_df = SOURCE_DFS[2].alias("products")
    history_df = SOURCE_DFS[3].alias("history")

    transformed_df = (
        orders_df
        .join(products_df, on="product_id", how="left")
        .join(history_df, on="customer_id", how="left")
        .withColumn(
            "order_net_amount",
            F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
        )
        .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
        .select(
            "order_id", "customer_id", "order_datetime", "modified_datetime",
            "product_id", "product_name", "product_category", "quantity", "unit_price",
            "discount", "order_net_amount", "order_status", "shipping_country",
            "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
        )
    )
    display(transformed_df)

# L. Load

## TARGET 1 — Curated Orders

The target follows **define → prep → checks → physical write → published read → profile/register → view**.

In [ ]:
if PIPELINE_SHOULD_RUN:
    TARGET = 1
    TARGET_NAME = "Curated Orders"
    TARGET_TARGET = "unified"
    TARGET_SCHEMA = "demo"
    TARGET_TABLE = "orders"
    TARGET_LOAD_STRATEGY = "scd1"
    TARGET_LOAD_PARAMETERS = {"key_columns": ["order_id"]}
    target_df = transformed_df

In [ ]:
if PIPELINE_SHOULD_RUN:
    target_prep = write_pipeline_prep(
        target_df,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE,
        load_strategy=TARGET_LOAD_STRATEGY,
        load_strategy_parameters=TARGET_LOAD_PARAMETERS,
        source_preps=[SOURCE_PREPS[1], SOURCE_PREPS[2], SOURCE_PREPS[3]],
    )
    TARGET_TABLE_ID = target_prep["target"]["table_id"]
    prepared_target_df = target_prep["df"].persist()

In [ ]:
if PIPELINE_SHOULD_RUN:
    target_checks = (
        [
            check_schema(table_id=TARGET_TABLE_ID, dataframe=prepared_target_df),
            check_dq(prepared_target_df, table_id=TARGET_TABLE_ID),
        ]
        if VALIDATE_DATA_CONTRACTS
        else []
    )
    if target_checks:
        display(target_checks[-1]["summary"])
    if not all(result["can_continue"] for result in target_checks):
        raise RuntimeError(f"A TARGET {TARGET} Guardrail blocked publication.")

### Physical write

`repartition_by=4` uses distributed Spark execution; it does not create Python threads or independent writers.

In [ ]:
if PIPELINE_SHOULD_RUN:
    write_lakehouse_table(
        prepared_target_df,
        target_prep["target"]["table_name"],
        target=target_prep["target"]["target"],
        schema=target_prep["target"]["schema"],
        mode=target_prep["mode"],
        options=target_prep["options"],
        load_strategy=target_prep["load_strategy"],
        load_strategy_parameters=target_prep["load_strategy_parameters"],
        processing_scope=target_prep["scope"],
        repartition_by=4,
    )
    prepared_target_df.unpersist()

### Published state and stored metadata

Read the physical target back before recording its canonical full-table profile.

In [ ]:
if PIPELINE_SHOULD_RUN:
    published_target_df = read_lakehouse_table(
        target_prep["target"]["table_name"],
        target=target_prep["target"]["target"],
        schema=target_prep["target"]["schema"],
        spark_session=spark,
    )
    target_profile = profile_and_register_table(
        published_target_df,
        profile_role="target",
        table=target_prep["target"],
        load_strategy=target_prep["load_strategy"],
        load_strategy_parameters=target_prep["load_strategy_parameters"],
    )
    display(target_profile)

    catalogue_widget["refresh"]()
    target_views = catalogue_widget["get_views"]()
    display(target_views["catalogue"].filter(F.col("table_id") == TARGET_TABLE_ID))
    display(target_views["profile"].filter(F.col("table_id") == TARGET_TABLE_ID))
    display(target_views["frequency"].filter(F.col("table_id") == TARGET_TABLE_ID))